In [2]:
from algorithms import (
    breadth_first_search_version_1, breadth_first_search_version_2,
    depth_first_search_version_1, depth_first_search_version_2,
    iterative_deepening_search_version_1, iterative_deepening_search_version_2,
    uniform_cost_search,
    greedy_search,
    a_star_search,
    iterative_deepening_a_star,
    simple_hill_climbing, steepest_ascent_hill_climbing, stochastic_hill_climbing, randomm_restart_hill_climbing,
    local_beam_search,
    simulated_annealing,
    sensorless_search_bfs_version,
    partial_obs_search_bfs_version,
    and_or_graph_search
)

In [3]:
import random

def gen_random_start_state(row_number=5, col_number=4):
    
    state = [[0] * col_number for _ in range(row_number)]

    # Vị trí robot: ô [0][0]
    state[0][0] = 'X'
 
    # 10% vật cản, 30% ô có rác, 60% ô trống
    for x in range(row_number):
        for y in range(col_number):
            if state[x][y] == 'X':
                continue

            r = random.random()
            if r < 0.1:
                state[x][y] = 2   
            elif r < 0.4:
                state[x][y] = 1  
 
    goal = [[0 if (cell == 'X' or cell == 1) else cell for cell in row] for row in state]
 
    return state, goal

In [7]:
import tkinter as tk
from tkinter import ttk, messagebox
import time

ALGORITHM = {
    "BFS version 1": 
    {
        "function": breadth_first_search_version_1,
        "type": "normal"
    },
    "BFS version 2": 
    {
        "function": breadth_first_search_version_2,
        "type": "normal"
    },
    "DFS version 1":
    {
        "function": depth_first_search_version_1,
        "type": "normal"
    }, 
    "DFS version 2": 
    {
        "function": depth_first_search_version_2,
        "type": "normal"
    }, 
    "IDS version 1": 
    {
        "function": iterative_deepening_search_version_1,
        "type": "normal"
    },
    "IDS version 2": 
    {
        "function": iterative_deepening_search_version_2,
        "type": "normal"
    }, 
    "UCS": 
    {
        "function": uniform_cost_search,
        "type": "normal"
    }, 
    "Greedy search": 
    {
        "function": greedy_search,
        "type": "normal"
    }, 
    "A*": 
    {
        "function": a_star_search,
        "type": "normal"
    },  
    "IDA*": 
    {
        "function": iterative_deepening_a_star,
        "type": "normal"
    }, 
    "Leo đồi đơn giản": 
    {
        "function": simple_hill_climbing,
        "type": "normal"
    }, 
    "Leo đồi dốc nhất": 
    {
        "function": steepest_ascent_hill_climbing,
        "type": "normal"
    }, 
    "Leo đồi ngẫu nhiên": 
    {
        "function": stochastic_hill_climbing,
        "type": "normal"
    }, 
    "Leo đồi khởi tạo ngẫu nhiên": 
    {
        "function": randomm_restart_hill_climbing,
        "type": "normal"
    }, 
    "Local beam search": 
    {
        "function": local_beam_search,
        "type": "normal"
    }, 
    "Mô phòng luyện kim": 
    {
        "function": simulated_annealing,
        "type": "normal"
    }, 
    "Môi trường không nhìn thấy": 
    {
        "function": sensorless_search_bfs_version,
        "type": "sensorless"
    },
    "Môi trường nhìn thấy 1 phần":
    {
        "function": partial_obs_search_bfs_version,
        "type": "partial_obs"
    },
    "And-Or Graph Search":
    {
        "function": and_or_graph_search,
        "type": "and-or"
    }
}
 
class VacuumGUI:

    def __init__(self, window: tk.Tk):
        self.window = window
        self.window.title("Máy hút bụi")
        self.window.geometry("1280x720+120+50")
        self.window.configure(bg="#EEF5FB")

        self.start =  [['X', 0 , 1 , 1],
                       [ 0 , 2 , 0 , 0],
                       [ 1 , 0 , 1 , 2],
                       [ 2 , 1 , 1 , 1],
                       [ 1 , 0 , 1 , 1]]

        self.goal = [[0, 0, 0, 0],
                     [0, 2, 0, 0],
                     [0, 0, 0, 2],
                     [2, 0, 0, 0],
                     [0, 0, 0, 0]]

        self.animating = False
 
        self.build_layout()
        self.create_grid()
        self.draw_state(self.start)
 

    def build_layout(self):
        
        # Left - Phần setting
        LEFT_FRAME_BG = "#E8F1FD"
        
        self.left_frame = tk.Frame(self.window, bg=LEFT_FRAME_BG, width=200, padx=4, pady=10)
        self.left_frame.pack(side="left", fill="y", padx=8, pady=8)
        self.left_frame.pack_propagate(False)
 
        tk.Label(self.left_frame, text="CÀI ĐẶT", 
                 fg="black", bg=LEFT_FRAME_BG, 
                 font=("Segoe UI", 20, "bold")).pack(anchor="center")
 
        # Chọn thuật toán 
        tk.Label(self.left_frame, text="Thuật toán", 
                 fg="black", bg=LEFT_FRAME_BG, 
                 font=("Segoe UI", 12, "bold")).pack(anchor="w")
 
        self.algorithm_var = tk.StringVar() 
        self.algorithm_box = ttk.Combobox(self.left_frame, 
                                          textvariable=self.algorithm_var, 
                                          values=list(ALGORITHM.keys()), 
                                          state="readonly", width=25)

        self.algorithm_box.current(0) 
        self.algorithm_box.pack(pady=4) 
 
        tk.Button(self.left_frame, text="BẮT ĐẦU", command=self.start_algorithm, fg="white", bg="navy", width=15).pack(pady=5)
        tk.Button(self.left_frame, text="ĐẶT LẠI", command=self.reset, fg="white", bg="saddlebrown", width=15).pack(pady=5)
 
        tk.Frame(self.left_frame, bg="blue", height=1).pack(pady=10,fill="x")

        # Tạo input ngẫu nhiên
        tk.Label(self.left_frame, text="Tạo input ngẫu nhiên", 
                 fg="black", bg=LEFT_FRAME_BG, 
                 font=("Segoe UI", 12, "bold")).pack(anchor="w")

        resize_frame = tk.Frame(self.left_frame, bg=LEFT_FRAME_BG)
        resize_frame.pack(padx=10, pady=8,fill="x")
 
        tk.Label(resize_frame, text="Hàng:", fg="black").grid(row=0, column=0, sticky="w")
        self.row_var = tk.IntVar(value=5)
        tk.Spinbox(resize_frame, from_=2, to=7, textvariable=self.row_var, width=4, fg="black").grid(row=0, column=1, padx=4)
 
        tk.Label(resize_frame, text="Cột:", fg="black").grid(row=1, column=0, sticky="w")
        self.col_var = tk.IntVar(value=4)
        tk.Spinbox(resize_frame, from_=2, to=7, textvariable=self.col_var, width=4, fg="black").grid(row=1, column=1, padx=4)
 
        tk.Button(self.left_frame, text="TẠO", command=self.randomize_map, bg="darkgreen", fg="white", width=15).pack(pady=6)
 
        tk.Frame(self.left_frame, bg="blue", height=1).pack(pady=10,fill="x")

        # Chú thích
        tk.Label(self.left_frame, text="Chú thích", bg=LEFT_FRAME_BG, font=("Segoe UI", 12, "bold")).pack(anchor="w")
        
        tk.Label(self.left_frame, text="🤖: Robot", bg=LEFT_FRAME_BG, fg="black").pack(padx=10, pady=(8, 5), anchor="w")
        tk.Label(self.left_frame, text="🍂: Rác", bg=LEFT_FRAME_BG, fg="black").pack(padx=10, pady=5, anchor="w")
        tk.Label(self.left_frame, text="🧱: Vật cản", bg=LEFT_FRAME_BG, fg="black").pack(padx=10, pady=5, anchor="w")
        tk.Label(self.left_frame, text="Ô trống: Không có rác và vật cản", bg=LEFT_FRAME_BG, fg="black").pack(padx=10, pady=5, anchor="w")
 

        # Center - phía trên: map, phía dưới: kết quả
        self.center_frame = tk.Frame(self.window, bg="#DCEEF8")
        self.center_frame.pack(side="left", fill="both", expand=True, padx=8, pady=8)
 
        self.map_frame = tk.LabelFrame(self.center_frame, text="Sơ đồ", 
                                       bg="#CFE7F5", fg="#0F172A",
                                       font=("Consolas", 13, "bold"),
                                       bd=2, labelanchor="n")
        self.map_frame.pack(fill="both", expand=True, padx=8, pady=4)
 
        self.result_frame = tk.LabelFrame(self.center_frame, text="Kết quả",
                                          bg="#CFE7F5", fg="#0F172A",
                                          font=("Consolas", 13, "bold"),
                                          bd=2, labelanchor="n")
        self.result_frame.pack(fill="x", padx=4, pady=4)
 
        self.result_text = tk.Text(self.result_frame, height=6,
                                   bg="white", fg="#2F4F5F",
                                   state="disabled")
        self.result_text.pack(fill="both", padx=4, pady=4)
 
        # Right - log
        self.log_frame = tk.LabelFrame(self.window, text="Log",
                                       bg="#E6F1F7", fg="#0E0C1B",
                                       font=("Consolas", 11, "bold"),
                                       bd=2, labelanchor="n")
        self.log_frame.pack(side="right", fill="y", padx=8, pady=8)
 
        self.log_scroll = tk.Scrollbar(self.log_frame)
        self.log_scroll.pack(side="right", fill="y")
 
        self.log_text = tk.Text(self.log_frame, width=33, 
                                bg="#E6F1F7", fg="#0F172A",
                                font=("Consolas", 9),
                                yscrollcommand=self.log_scroll.set,
                                state="disabled")
        self.log_text.pack(fill="both", expand=True, padx=4, pady=4)
        self.log_scroll.config(command=self.log_text.yview)
 

    def create_grid(self):
        for widget in self.map_frame.winfo_children():
            widget.destroy()
 
        self.cells = []
        row_number = len(self.start)
        col_number = len(self.start[0])
 
        self.map_frame.grid_columnconfigure(list(range(col_number+2)), weight=1)
        self.map_frame.grid_rowconfigure(list(range(row_number+2)), weight=1)
 
        for i in range(row_number):
            row = []
            for j in range(col_number):
                cell = tk.Label(self.map_frame,
                                width=4, height=2,
                                font=("Segoe UI Emoji", 18))
                cell.grid(row=i+1, column=j+1, padx=3, pady=3, sticky="nsew")
                row.append(cell)
            self.cells.append(row)
 

    def draw_state(self, state):
        cell_dict = {
            "X": {"icon" : "🤖", "color" : "#C7E4F4"},
            0: {"icon" : "", "color" : "#EAF4FA"},
            1: {"icon" : "🍂", "color" : "#DCEAD7"},
            2: {"icon" : "🧱", "color" : "#C8D3DC"}
        }

        for i in range(len(state)):
            for j in range(len(state[0])):
                v = state[i][j]
                self.cells[i][j].config(text=cell_dict.get(v).get("icon"), bg=cell_dict.get(v).get("color"))


    def animate_path(self, path):
        idx = 0
        algo_name = self.algorithm_var.get()
        type_algo = ALGORITHM.get(algo_name)["type"]
        for node in path:
            if not self.animating:
                return

            if type_algo == "normal":
                step_label = "Khởi tạo" if node.action is None else node.action
                self.write_log(f"Bước: {idx}\nCost = {node.path_cost}\n{step_label}")
                self.draw_state(node.state)
            
                for row in node.state:
                    self.write_log("\t" + " ".join(map(str, row)))
                self.window.update()   
                time.sleep(0.6) 
            elif type_algo in ["sensorless", "partial_obs"]:
                step_label = "Belief state" if node.action is None else node.action
                self.write_log(f"Bước: {idx} -- {step_label}")
                self.draw_state(node.state[0])
                self.write_log(f" -*- Số trạng thái: {len(node.state)}")
                for i, state in enumerate(node.state):
                    self.write_log(f"  [State {i+1}]")
                    for row in state:
                        self.write_log("\t" + " ".join(map(str, row)))
                self.window.update()
                time.sleep(0.8)

            idx += 1

        self.write_log("Phòng đã được dọn sạch!!!")
        self.animating = False


    def get_result(self, path):
        s = f"Số bước: {len(path)-1}\nChuỗi hành động: "
        for node in path:
            step_action = "Khởi tạo" if node.action is None else node.action
            s += step_action + " → "
        
        s += "Goal"
        return s
        

    def start_algorithm(self):
        if self.animating:
            return
 
        algo_name = self.algorithm_var.get()
        run_algo = ALGORITHM.get(algo_name)["function"]
        type_algo = ALGORITHM.get(algo_name)["type"]
        if run_algo is None:
            return

        if type_algo == "and-or":
            self.run_and_or_graph_search(self.start, self.goal)
            return
        self.clear_texts()
        if  type_algo == "normal":
            self.draw_state(self.start)
        else:
            self.draw_state(self.goal)
 
        self.write_log(f"Thuật toán: {algo_name}")
        self.write_log("Đang tìm đường đi...\n")

        start_time = time.perf_counter()

        if run_algo == local_beam_search:
            self.ask_beam_width()
            path = run_algo(self.start, self.goal, self.beam_width)
        elif type_algo == "normal":
            path = run_algo(self.start, self.goal)
        elif type_algo == "sensorless":
            path = run_algo(self.goal)
        elif type_algo == "partial_obs":
            self.init_for_partial_obs()
            path = run_algo(self.start, self.goal)

        end_time = time.perf_counter()
        execution_time = end_time - start_time
 
        if path is None:
            self.write_log("Không tìm thấy đường đi!!!")
            self.write_result("Không tìm thấy đường đi!!!")
            return
        
        self.write_result(f"Thời gian tìm kiếm: {execution_time:.5f} s\n" + self.get_result(path))
        self.write_log("Đã tìm thấy cách di chuyển. \nBắt đầu hành động...\n")
        self.animating = True
        self.animate_path(path)
 

    def reset(self):
        algo_name = self.algorithm_var.get()
        type_algo = ALGORITHM.get(algo_name)["type"]

        self.animating = False
        if type_algo == "normal":
            self.draw_state(self.start)
        elif type_algo == "sensorless":
            self.draw_state(self.goal)
        elif type_algo == "partial_obs":
            self.init_for_partial_obs()
            self.draw_state(self.start)

        self.clear_texts()
 

    def randomize_map(self):
        self.animating = False
        algo_name = self.algorithm_var.get()
        type_algo = ALGORITHM.get(algo_name)["type"]
        if type_algo == "partial_obs":
            self.init_for_partial_obs()
            tk.messagebox.showerror("Thông báo", "Thuật toán được cố định, không thể random!")
        else:
            row_number = self.row_var.get()
            col_number = self.col_var.get()
            self.start, self.goal = gen_random_start_state(row_number, col_number)

        self.create_grid()
        self.draw_state(self.start)
        self.clear_texts()


    def write_log(self, msg):
        self.log_text.config(state="normal")
        self.log_text.insert(tk.END, msg + "\n")
        self.log_text.see(tk.END)
        self.log_text.config(state="disabled")
 

    def write_result(self, msg):
        self.result_text.config(state="normal")
        self.result_text.insert(tk.END, msg)
        self.result_text.see(tk.END)
        self.result_text.config(state="disabled")
    

    def clear_texts(self):
        self.log_text.config(state="normal")
        self.result_text.config(state="normal")
        self.log_text.delete("1.0", tk.END)
        self.result_text.delete("1.0", tk.END)
        self.log_text.config(state="disabled")
        self.result_text.config(state="disabled")
    
    def ask_beam_width(self):
        popup = tk.Toplevel(self.window)
        popup.title("Local Beam Search")
        popup.geometry("300x150+600+300")
        popup.resizable(False, False)
        popup.grab_set()

        tk.Label(popup, text="Nhập Beam Width (k):", font=("Segoe UI", 12)).pack(pady=10)

        beam_entry = tk.Entry(popup, width=20, font=("Segoe UI", 12),justify="center")
        beam_entry.pack(pady=5)

        beam_entry.insert(0, "3")

        def submit():
            value = beam_entry.get().strip()

            if not value.isdigit():
                tk.messagebox.showerror( "Lỗi", "k phải là số nguyên dương!")
                return

            beam_width = int(value)

            if beam_width <= 0 or beam_width > 4:
                tk.messagebox.showerror("Lỗi", "Định dạng đúng: 0 < k <= 4")
                return

            popup.destroy()
            self.beam_width = beam_width

        tk.Button(popup, text="OK", command=submit, width=10, bg="darkgreen", fg="white").pack(pady=10)
        beam_entry.focus()

        self.window.wait_window(popup)
    
    def init_for_partial_obs(self):
        self.start = [[0, 0, 0],
                      [1, 0, 0],
                      [1, 1, 0]] 

        self.goal = [[0, 0, 0],
                     [0, 0, 0],
                     [0, 0, 2]] 
    
    def run_and_or_graph_search(self, start_map, goal_map):

        cell_dict = {
            "X": {"icon" : "🤖", "color" : "#C7E4F4"},
            0: {"icon" : "  ", "color" : "#EAF4FA"},
            1: {"icon" : "🍂", "color" : "#DCEAD7"},
            2: {"icon" : "🧱", "color" : "#C8D3DC"}
        }

        node_matrix_storage = {}

        def populate_tree_rich(tree, parent, plan, current_matrix, count=1):
            if plan is None:
                node_id = tree.insert(parent, "end", text="FAILURE", tags=("failure",))
                node_matrix_storage[node_id] = current_matrix
                return count
                
            if plan == []: 
                node_id = tree.insert(parent, "end", text="GOAL", tags=("goal",))
                node_matrix_storage[node_id] = current_matrix
                return count

            if isinstance(plan, tuple):
                action, next_plan = plan
                node_id = tree.insert(parent, "end", text=f"Chọn hành động: {action}", tags=("choice",))
                node_matrix_storage[node_id] = current_matrix

                count = populate_tree_rich(tree, node_id, next_plan, current_matrix, count)

            elif isinstance(plan, dict):
                temp = count
                count += len(plan)
                for key, value in plan.items():
                    actual_action, next_matrix = key
                    
                    node_text = f"[{temp}] Nếu: {actual_action}"
                    node_id = tree.insert(parent, "end", text=node_text, tags=("environment",))

                    node_matrix_storage[node_id] = next_matrix
                    
                    count = populate_tree_rich(tree, node_id, value, next_matrix, count)
                    temp += 1
                    
            return count

        def update_matrix_view(canvas, matrix):

            canvas.delete("all")
            if not matrix:
                canvas.create_text(150, 150, text="Chọn một nút trên cây\nđể xem bản đồ tương ứng", font=("Helvetica", 11, "italic"), fill="#7f8c8d", justify=tk.CENTER)
                return
                
            rows = len(matrix)
            cols = len(matrix[0])
            
            cell_width = 300 // cols
            cell_height = 300 // rows
            
            for r in range(rows):
                for c in range(cols):
                    cell_value = matrix[r][c]
                    cell_info = cell_dict.get(cell_value, {"icon": str(cell_value), "color": "#FFFFFF"})
                    
                    x1, y1 = c * cell_width, r * cell_height
                    x2, y2 = x1 + cell_width, y1 + cell_height
                    
                    canvas.create_rectangle(x1, y1, x2, y2, fill=cell_info["color"], outline="#BDC3C7", width=1)
                    
                    canvas.create_text((x1 + x2) / 2, (y1 + y2) / 2, text=cell_info["icon"], font=("Segoe UI Emoji", 24))
                    
        and_or_window = tk.Toplevel(self.window)

        def on_close():
            and_or_window.destroy()

        and_or_window.protocol("WM_DELETE_WINDOW", on_close)

        and_or_window.title("And - Or Graph Search")
        and_or_window.geometry("900x650+200+100") 

        main_container = tk.Frame(and_or_window, bg="#F8F9FA")
        main_container.pack(fill=tk.BOTH, expand=True, padx=15, pady=5)

        left_frame = tk.LabelFrame(main_container, text=" Cây kết quả ", font=("Helvetica", 11, "bold"), fg="black", padx=5, pady=5)
        left_frame.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

        scrollbar = tk.Scrollbar(left_frame)
        scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

        tree = ttk.Treeview(left_frame, yscrollcommand=scrollbar.set)
        tree.pack(fill=tk.BOTH, expand=True)
        scrollbar.config(command=tree.yview)
            
        tree.column("#0", width=450, minwidth=350)
        tree.heading('#0', text='Chi tiết')

        tree.tag_configure("choice", foreground="#2980B9", font=("Helvetica", 10, "bold"))
        tree.tag_configure("environment", foreground="#D35400", background="#FFF5EE")
        tree.tag_configure("goal", foreground="#27AE60", background="#E8F8F5", font=("Helvetica", 10, "bold"))
        tree.tag_configure("failure", foreground="#C0392B", background="#FDEDEC", font=("Helvetica", 10, "bold"))

        style = ttk.Style()
        style.configure("Treeview", font=("Segoe UI Emoji", 10), rowheight=28)

        result_plan = and_or_graph_search(start_map, goal_map)
        populate_tree_rich(tree, "", result_plan, start_map)

        def open_all_nodes(parent_node=""):
            for child in tree.get_children(parent_node):
                tree.item(child, open=True)
                open_all_nodes(child)
        open_all_nodes()

        right_frame = tk.LabelFrame(main_container, text=" MAP ", font=("Helvetica", 11, "bold"), fg="black", padx=20, pady=15)
        right_frame.pack(side=tk.RIGHT, fill=tk.BOTH, padx=(15, 0))

        map_canvas = tk.Canvas(right_frame, width=300, height=300, bg="#FFFFFF", highlightthickness=1, highlightbackground="#BDC3C7")
        map_canvas.pack(pady=5)
            
        update_matrix_view(map_canvas, start_map)

        def on_tree_select(event):
            selected_item = tree.focus()
            if selected_item in node_matrix_storage:
                matrix = node_matrix_storage[selected_item]
                update_matrix_view(map_canvas, matrix)

        tree.bind("<<TreeviewSelect>>", on_tree_select)

        and_or_window.transient(self.window)
        and_or_window.grab_set()


if __name__ == "__main__":
    window = tk.Tk()
    app = VacuumGUI(window)
    window.mainloop()